In [63]:
import importlib
import inspect
import pkgutil
import typing

import pandas as pd
from IPython.display import Markdown, display
from langchain.chains import RetrievalQA
from langchain.chains.combine_documents.base import BaseCombineDocumentsChain
from langchain.chains.combine_documents.stuff import StuffDocumentsChain
from langchain.chains.llm import LLMChain
from langchain.memory import ChatMessageHistory
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain.vectorstores import Chroma, VectorStore
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader
from langchain_core.documents import Document
from langchain_core.language_models.base import LanguageModelInput
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langchain_core.output_parsers import CommaSeparatedListOutputParser, JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.stores import BaseStore
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TextSplitter,
)

# TYPE TETRIS 2 
<details>
<summary><b>Clique para expandir: Minha Seção Oculta</b></summary>

| | investiga | grafo | uso |
|---|---|---|---|
| `probe(obj)` | contratos de tipo (assinaturas, Unions) | grafo de **tipos** — cadeia sequencial | `probe=ChatOpenAI().invoke` `probe=ChatOpenAI` |
| `target(obj)` | um objeto que já existe | grafo de **instâncias** | `target=ChatOpenAI()` `target=ChatOpenAI().invoke()`

In [64]:
# %%catch — wraps a cell's execution so an exception can never halt "Run All".
from IPython.core.magic import register_cell_magic


@register_cell_magic
def catch(line, cell):
    result = get_ipython().run_cell(cell)
    if not result.success:
        print("⚠️ cell failed — continuing to the next cell")

## Helper visual

`_align_left` só formata o DataFrame pra ficar alinhado à esquerda (mesmo
truque do notebook original). Usado por `probe` e por `target`, sem log
nenhum por trás.

In [65]:
def _align_left(df: pd.DataFrame):
    styler = (
        df.style.format(escape="html")
        .set_properties(**{"text-align": "left"})
        .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}])
    )
    if "required" in df.columns:
        styler = styler.apply(
            lambda row: ["color: orange" if row["required"] else "" for _ in row],
            axis=1,
        )
    return styler

## `probe(obj, label=None)`

Um passo da cadeia de tipos — sem acumular histórico na tela: cada chamada
mostra só o próprio resultado, embaixo da célula que a disparou. Detecta
sozinho se `obj` é:
- um **construto do `typing`** (Union, Sequence[...], etc.) — mostra o **tipo
  completo resolvido** (repr integral, sem corte, num bloco de código) e
  depois os branches individuais via `typing.get_args`, um por linha;
- um **callable** (função, método, classe) — mostra os parâmetros via
  `inspect.signature`, no mesmo formato de tabela do notebook original
  (`name`, `kind`, `default`, `required`, `annotation`);
- outra coisa — mostra `repr`/`type` como fallback.

Você decide manualmente qual nome/branch do resultado vira o próximo `probe(...)`.

In [66]:
def probe(obj, label: str | None = None):
    label = label or getattr(obj, "__name__", repr(obj))
    args = typing.get_args(obj)

    if args:
        origin = typing.get_origin(obj)
        kind = f"typing construct ({getattr(origin, '__name__', repr(origin))})"
        display(Markdown(f"**{label}** — _{kind}_"))
        display(Markdown(f"tipo completo:\n```\n{obj!r}\n```"))
        df = pd.DataFrame(
            [
                {"index": i, "arg": repr(a), "type_of_arg": type(a).__name__}
                for i, a in enumerate(args)
            ]
        )
    elif callable(obj):
        kind = "callable signature"
        display(Markdown(f"**{label}** — _{kind}_"))
        sig = inspect.signature(obj)
        df = pd.DataFrame(
            [
                {
                    "name": name,
                    "kind": p.kind.name,
                    "default": repr(p.default),
                    "required": p.default is inspect.Parameter.empty
                    and p.kind
                    not in (inspect.Parameter.VAR_POSITIONAL, inspect.Parameter.VAR_KEYWORD),
                    "annotation": str(p.annotation),
                }
                for name, p in sig.parameters.items()
            ]
        )
    else:
        kind = "raw object"
        display(Markdown(f"**{label}** — _{kind}_"))
        df = pd.DataFrame([{"repr": repr(obj), "type": type(obj).__name__}])

    display(_align_left(df) if not df.empty else df)

## `target(obj, label=None)`

Mesmo mecanismo do `probe`: uma chamada, um resultado, sem histórico. A
diferença é que aqui uma chamada sempre produz **4 blocos** de uma vez, um
embaixo do outro, todos sobre o mesmo `obj`:

- **abstração** — `inspect.isabstract(cls)` + `cls.__abstractmethods__`.
  Responde uma pergunta que nem `probe` nem o resto do `target` respondem:
  essa classe pode ser instanciada de verdade, ou é só um contrato com
  métodos pendentes (ex: `combine_docs` em `BaseCombineDocumentsChain`)?
  Sem isso, dá pra confundir "método existe no `dir()`" com "método tem
  implementação" — os dois aparecem igual pro `dir()`, só o
  `__abstractmethods__` distingue.
- **construção** — assinatura do construtor da classe (o que você precisaria
  passar pra construir um objeto desses).
- **dados** — atributos não-chamáveis da instância (o estado atual).
- **métodos** — atributos chamáveis da instância (o comportamento
  disponível).

Uma célula = um objeto de investigação. Se depois de ver os 4 blocos você
quiser investigar outra coisa (ex: um atributo que apareceu em "dados"), isso
vira uma célula nova com um novo `target(...)`.

In [67]:
def _describe_valor(value):
    r = repr(value)
    if len(r) <= 60:
        return r
    if isinstance(value, str):
        return f"str ({len(value)} chars)"
    if isinstance(value, dict):
        return f"dict ({len(value)} keys)"
    if isinstance(value, (list, tuple, set)):
        return f"{type(value).__name__} ({len(value)} items)"
    return f"{type(value).__name__} (repr longo: {len(r)} chars)"


def target(obj, label: str | None = None):
    cls = obj if inspect.isclass(obj) else type(obj)
    label = label or cls.__name__

    # abstração
    is_abstract = inspect.isabstract(cls)
    pending = sorted(cls.__abstractmethods__) if is_abstract else []
    status = (
        f"abstrata — métodos pendentes: {', '.join(pending)}"
        if is_abstract
        else "concreta (instanciável)"
    )
    display(Markdown(f"**{label}** — _{status}_"))

    # construção
    sig = inspect.signature(cls)
    df_construcao = pd.DataFrame(
        [
            {
                "name": name,
                "kind": p.kind.name,
                "default": repr(p.default),
                "required": p.default is inspect.Parameter.empty
                and p.kind not in (inspect.Parameter.VAR_POSITIONAL, inspect.Parameter.VAR_KEYWORD),
                "annotation": str(p.annotation),
            }
            for name, p in sig.parameters.items()
        ]
    )
    display(Markdown(f"**{label}** — _construção (constructor signature)_"))
    display(_align_left(df_construcao) if not df_construcao.empty else df_construcao)

    # dados
    rows = []
    for name in dir(obj):
        if name.startswith("_"):
            continue
        try:
            value = getattr(obj, name)
        except Exception:
            continue
        if callable(value):
            continue
        rows.append({"name": name, "resumo": _describe_valor(value), "print[:40]": str(value)[:40]})
    df_dados = pd.DataFrame(rows, columns=["name", "resumo", "print[:40]"])
    if not df_dados.empty:
        df_dados = df_dados.sort_values("name").reset_index(drop=True)
    display(Markdown(f"**{label}** — _dados da instância_"))
    display(_align_left(df_dados) if not df_dados.empty else df_dados)

    # métodos
    rows = []
    for name in dir(obj):
        if name.startswith("_"):
            continue
        try:
            value = getattr(obj, name)
        except Exception:
            continue
        if not callable(value):
            continue
        self_repr = type(value.__self__).__name__ if hasattr(value, "__self__") else "?"
        is_pending = name in cls.__abstractmethods__ if is_abstract else False
        rows.append({"name": name, "method": f"{self_repr}.{name}", "abstract": is_pending})
    df_metodos = pd.DataFrame(rows, columns=["name", "method", "abstract"])
    if not df_metodos.empty:
        df_metodos = df_metodos.sort_values("name").reset_index(drop=True)
    display(Markdown(f"**{label}** — _métodos da instância_"))
    display(_align_left(df_metodos) if not df_metodos.empty else df_metodos)

## `full_init_params(cls)`

Resolve o buraco negro do `**kwargs`: nem `probe` nem `target` conseguem ver o
que uma subclasse repassa pro `__init__` da classe-mãe via
`super().__init__(**kwargs)` — a assinatura da subclasse só mostra
`**kwargs: Any`, sem dizer o que cabe lá dentro.

`full_init_params` percorre `inspect.getmro(cls)` (a cadeia de herança) e,
pra cada ancestral que define seu **próprio** `__init__` (não herdado — via
`"__init__" in vars(klass)`), extrai os parâmetros reais desse `__init__`.
Resultado: uma tabela única com todo parâmetro que a classe aceita, direta
ou indiretamente, junto com o nome da classe de onde ele realmente vem
(coluna `from`).

Use quando `probe(cls)` ou a tabela de "construção" do `target(cls)`
terminarem em `**kwargs: Any` e você precisar saber o que está escondido
ali.

In [68]:
def full_init_params(cls):
    seen = {}
    for klass in inspect.getmro(cls):
        if "__init__" not in vars(klass):  # só __init__ definido NESSA classe, não herdado
            continue
        for name, p in inspect.signature(klass.__init__).parameters.items():
            if name in ("self", "args", "kwargs"):
                continue
            seen.setdefault(
                name, (klass.__name__, p)
            )  # primeira ocorrência = mais específica na MRO

    df = pd.DataFrame(
        [
            {
                "name": name,
                "from": owner,
                "kind": p.kind.name,
                "default": repr(p.default),
                "required": p.default is inspect.Parameter.empty
                and p.kind not in (inspect.Parameter.VAR_POSITIONAL, inspect.Parameter.VAR_KEYWORD),
                "annotation": str(p.annotation),
            }
            for name, (owner, p) in seen.items()
        ]
    )
    display(Markdown(f"**{cls.__name__}** — _full init params (via MRO)_"))
    display(_align_left(df) if not df.empty else df)

### `**kwargs` em métodos — quando `full_init_params` não basta

1. Método (classmethod ou de instância) também aceita `**kwargs` — e é até mais comum que em função solta (ex: `from_documents`/`from_texts` do LangChain).
2. Diferente do caso de classe, aqui não tem MRO: o método só repassa `**kwargs` pra dentro do próprio corpo — uma chamada arbitrária, não uma cadeia formal.
3. Padrão "factory" (`from_x`) costuma desembocar em `cls(**kwargs)`, isto é, no `__init__` da própria classe.
4. Nesse caso, `full_init_params(cls)` ainda serve — só que aplicado na classe, não no método.
5. Não é garantido: hops intermediários (ex: `from_texts` dentro de `from_documents`) podem extrair parâmetros nomeados que não aparecem nem na assinatura do método nem em `full_init_params`.

**Fluxo rápido:**
1. `probe(classe.metodo)` — vê a assinatura explícita + o `**kwargs` opaco.
2. Se suspeitar de um factory que desemboca no construtor: `full_init_params(classe)` — vê o que o `__init__` final aceita.
3. Se tiver hop intermediário no meio: `inspect.getsource(classe.metodo)` — único jeito de confirmar automaticamente o que cada hop extrai.

## `siblings(cls, label=None)`

Resolve "quem implementa isso?" — o que nem `probe` nem `target` fazem
sozinhos, porque os dois só investigam o objeto que você já aponta pra
eles; nenhum faz busca reversa tipo "liste as subclasses de `X` que
existem no pacote".

Sobe da classe pro pacote-pai (via `cls.__module__`, cortando o último
segmento) e cobre dois modos de busca ali dentro:

- **Modo A — `__all__`**: a lista que o autor do pacote declarou como API
  pública. Confiável quando existe, mas nem todo pacote a mantém completa
  — ex: `langchain.chains.combine_documents` só exporta funções auxiliares
  em `__all__`, nunca as classes de chain (`StuffDocumentsChain` etc.).
- **Modo B — `pkgutil.iter_modules` + `issubclass`**: ignora `__all__` e lê
  a estrutura real de arquivos do pacote. Importa cada submódulo-irmão e
  filtra só as classes que (a) foram **definidas** ali — não reimportadas
  de outro lugar — e (b) são de fato subclasse de `cls`. Sempre funciona,
  mesmo quando o Modo A não ajuda em nada.

**Limitação:** os dois modos partem do pressuposto de que a classe-base
mora dentro de um **subpacote dedicado** (uma pasta com vários `.py`, um
por implementação — como `combine_documents/`). Quando a classe-base é só
um módulo solto dentro de um pacote gigante (ex: `BaseStore`, que vive em
`langchain_core/stores.py`, dentro do enorme `langchain_core`), `siblings`
sobe pro pacote errado — grande demais, sem sinal útil. Nesse caso, ainda
vale a pista manual (import já existente em algum notebook, doc oficial,
etc.) — ver S2 abaixo.

In [69]:
def siblings(cls, label: str | None = None):
    label = label or cls.__name__
    leaf_mod_name = cls.__module__
    pkg_name = leaf_mod_name.rsplit(".", 1)[0] if "." in leaf_mod_name else leaf_mod_name
    pkg = importlib.import_module(pkg_name)

    display(Markdown(f"**{label}** — _pacote: `{pkg.__name__}`_"))

    # Modo A — __all__ (API pública declarada pelo autor do pacote)
    display(Markdown("Modo A — `__all__`:"))
    exported = getattr(pkg, "__all__", None)
    df_a = pd.DataFrame({"name": sorted(exported)}) if exported else pd.DataFrame(columns=["name"])
    display(_align_left(df_a) if not df_a.empty else df_a)

    # Modo B — módulos irmãos (estrutura real do pacote) + filtro por issubclass
    display(Markdown(f"Modo B — subclasses reais de `{label}` nos módulos irmãos:"))
    rows = []
    if hasattr(pkg, "__path__"):
        for modinfo in pkgutil.iter_modules(pkg.__path__):
            try:
                submod = importlib.import_module(f"{pkg.__name__}.{modinfo.name}")
            except Exception:
                continue
            for name, obj in inspect.getmembers(submod, inspect.isclass):
                if obj.__module__ != submod.__name__:
                    continue  # só classes definidas NESSE módulo, não reimportadas
                if obj is cls or not issubclass(obj, cls):
                    continue
                rows.append({"module": modinfo.name, "class": name})
    df_b = pd.DataFrame(rows, columns=["module", "class"])
    if not df_b.empty:
        df_b = df_b.sort_values(["module", "class"]).reset_index(drop=True)
    display(_align_left(df_b) if not df_b.empty else df_b)

# `PROBE`

In [77]:
_ = ChatMessageHistory.add_message

probe(_, label=str(_))

**<function InMemoryChatMessageHistory.add_message at 0x11fcd2e80>** — _callable signature_

,name,kind,default,required,annotation
0,self,POSITIONAL_OR_KEYWORD,<class 'inspect._empty'>,True,<class 'inspect._empty'>
1,message,POSITIONAL_OR_KEYWORD,<class 'inspect._empty'>,True,BaseMessage


## P2

In [78]:
_ = SystemMessage

probe(_, label=str(_))

**<class 'langchain_core.messages.system.SystemMessage'>** — _callable signature_

,name,kind,default,required,annotation
0,content,POSITIONAL_OR_KEYWORD,<class 'inspect._empty'>,True,"typing.Union[str, list[typing.Union[str, dict]]]"
1,additional_kwargs,KEYWORD_ONLY,<factory>,False,<class 'dict'>
2,response_metadata,KEYWORD_ONLY,<factory>,False,<class 'dict'>
3,type,KEYWORD_ONLY,'system',False,typing.Literal['system']
4,name,KEYWORD_ONLY,None,False,typing.Optional[str]
5,id,KEYWORD_ONLY,None,False,"typing.Annotated[typing.Optional[str], _PydanticGeneralMetadata(coerce_numbers_to_str=True)]"
6,kwargs,VAR_KEYWORD,<class 'inspect._empty'>,False,typing.Any


## P3

In [79]:
_ = BaseMessage

probe(_, label=str(_))

**<class 'langchain_core.messages.base.BaseMessage'>** — _callable signature_

,name,kind,default,required,annotation
0,content,POSITIONAL_OR_KEYWORD,<class 'inspect._empty'>,True,"Union[str, list[Union[str, dict]]]"
1,additional_kwargs,KEYWORD_ONLY,<factory>,False,<class 'dict'>
2,response_metadata,KEYWORD_ONLY,<factory>,False,<class 'dict'>
3,type,KEYWORD_ONLY,<class 'inspect._empty'>,True,<class 'str'>
4,name,KEYWORD_ONLY,None,False,typing.Optional[str]
5,id,KEYWORD_ONLY,None,False,"typing.Annotated[typing.Optional[str], _PydanticGeneralMetadata(coerce_numbers_to_str=True)]"
6,kwargs,VAR_KEYWORD,<class 'inspect._empty'>,False,typing.Any


In [73]:
_ = LLMChain

probe(_, label=str(_))

**<class 'langchain.chains.llm.LLMChain'>** — _callable signature_

,name,kind,default,required,annotation
0,args,VAR_POSITIONAL,<class 'inspect._empty'>,False,typing.Any
1,name,KEYWORD_ONLY,None,False,typing.Optional[str]
2,memory,KEYWORD_ONLY,None,False,typing.Optional[langchain_core.memory.BaseMemory]
3,callbacks,KEYWORD_ONLY,None,False,"typing.Union[list[langchain_core.callbacks.base.BaseCallbackHandler], langchain_core.callbacks.base.BaseCallbackManager, NoneType]"
4,verbose,KEYWORD_ONLY,<factory>,False,<class 'bool'>
5,tags,KEYWORD_ONLY,None,False,typing.Optional[list[str]]
6,metadata,KEYWORD_ONLY,None,False,"typing.Optional[dict[str, typing.Any]]"
7,callback_manager,KEYWORD_ONLY,None,False,typing.Optional[langchain_core.callbacks.base.BaseCallbackManager]
8,prompt,KEYWORD_ONLY,<class 'inspect._empty'>,True,<class 'langchain_core.prompts.base.BasePromptTemplate'>
9,llm,KEYWORD_ONLY,<class 'inspect._empty'>,True,"typing.Union[langchain_core.runnables.base.Runnable[typing.Union[langchain_core.prompt_values.PromptValue, str, collections.abc.Sequence[typing.Union[langchain_core.messages.base.BaseMessage, list[str], tuple[str, str], str, dict[str, typing.Any]]]], str], langchain_core.runnables.base.Runnable[typing.Union[langchain_core.prompt_values.PromptValue, str, collections.abc.Sequence[typing.Union[langchain_core.messages.base.BaseMessage, list[str], tuple[str, str], str, dict[str, typing.Any]]]], langchain_core.messages.base.BaseMessage]]"


# `TARGET`

In [74]:
_ = ChatMessageHistory()

target(_, label=(str(_)))

**InMemoryChatMessageHistory** — _concreta (instanciável)_

**InMemoryChatMessageHistory** — _construção (constructor signature)_

,name,kind,default,required,annotation
0,messages,KEYWORD_ONLY,<factory>,False,list[langchain_core.messages.base.BaseMessage]


/var/folders/pf/7lwsqjw92g96dl5sfdckf9_r0000gn/T/ipykernel_5028/2879068932.py:52: PydanticDeprecatedSince211: Accessing the 'model_computed_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  value = getattr(obj, name)
/var/folders/pf/7lwsqjw92g96dl5sfdckf9_r0000gn/T/ipykernel_5028/2879068932.py:52: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  value = getattr(obj, name)


**InMemoryChatMessageHistory** — _dados da instância_

,name,resumo,print[:40]
0,messages,[],[]
1,model_computed_fields,{},{}
2,model_config,{},{}
3,model_extra,None,None
4,model_fields,dict (1 keys),{'messages': FieldInfo(annotation=list[B
5,model_fields_set,set(),set()


/var/folders/pf/7lwsqjw92g96dl5sfdckf9_r0000gn/T/ipykernel_5028/2879068932.py:70: PydanticDeprecatedSince211: Accessing the 'model_computed_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  value = getattr(obj, name)
/var/folders/pf/7lwsqjw92g96dl5sfdckf9_r0000gn/T/ipykernel_5028/2879068932.py:70: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  value = getattr(obj, name)


**InMemoryChatMessageHistory** — _métodos da instância_

,name,method,abstract
0,aadd_messages,InMemoryChatMessageHistory.aadd_messages,False
1,aclear,InMemoryChatMessageHistory.aclear,False
2,add_ai_message,InMemoryChatMessageHistory.add_ai_message,False
3,add_message,InMemoryChatMessageHistory.add_message,False
4,add_messages,InMemoryChatMessageHistory.add_messages,False
5,add_user_message,InMemoryChatMessageHistory.add_user_message,False
6,aget_messages,InMemoryChatMessageHistory.aget_messages,False
7,clear,InMemoryChatMessageHistory.clear,False
8,construct,ModelMetaclass.construct,False
9,copy,InMemoryChatMessageHistory.copy,False


## T2

In [75]:
_ = BaseMessage()

target(_, label=(str(_)))

TypeError: BaseMessage.__init__() missing 1 required positional argument: 'content'

## T3

In [ ]:
_ = LLMChain

target(_, label=(str(_)))

**<class 'langchain.chains.llm.LLMChain'>** — _concreta (instanciável)_

**<class 'langchain.chains.llm.LLMChain'>** — _construção (constructor signature)_

,name,kind,default,required,annotation
0,args,VAR_POSITIONAL,<class 'inspect._empty'>,False,typing.Any
1,name,KEYWORD_ONLY,None,False,typing.Optional[str]
2,memory,KEYWORD_ONLY,None,False,typing.Optional[langchain_core.memory.BaseMemory]
3,callbacks,KEYWORD_ONLY,None,False,"typing.Union[list[langchain_core.callbacks.base.BaseCallbackHandler], langchain_core.callbacks.base.BaseCallbackManager, NoneType]"
4,verbose,KEYWORD_ONLY,<factory>,False,<class 'bool'>
5,tags,KEYWORD_ONLY,None,False,typing.Optional[list[str]]
6,metadata,KEYWORD_ONLY,None,False,"typing.Optional[dict[str, typing.Any]]"
7,callback_manager,KEYWORD_ONLY,None,False,typing.Optional[langchain_core.callbacks.base.BaseCallbackManager]
8,prompt,KEYWORD_ONLY,<class 'inspect._empty'>,True,<class 'langchain_core.prompts.base.BasePromptTemplate'>
9,llm,KEYWORD_ONLY,<class 'inspect._empty'>,True,"typing.Union[langchain_core.runnables.base.Runnable[typing.Union[langchain_core.prompt_values.PromptValue, str, collections.abc.Sequence[typing.Union[langchain_core.messages.base.BaseMessage, list[str], tuple[str, str], str, dict[str, typing.Any]]]], str], langchain_core.runnables.base.Runnable[typing.Union[langchain_core.prompt_values.PromptValue, str, collections.abc.Sequence[typing.Union[langchain_core.messages.base.BaseMessage, list[str], tuple[str, str], str, dict[str, typing.Any]]]], langchain_core.messages.base.BaseMessage]]"


**<class 'langchain.chains.llm.LLMChain'>** — _dados da instância_

,name,resumo,print[:40]
0,InputType,<property object at 0x11f9d3330>,<property object at 0x11f9d3330>
1,OutputType,<property object at 0x11fa79580>,<property object at 0x11fa79580>
2,config_specs,<property object at 0x11fa79620>,<property object at 0x11fa79620>
3,input_keys,<property object at 0x11fd6ecf0>,<property object at 0x11fd6ecf0>
4,input_schema,<property object at 0x11fa79530>,<property object at 0x11fa79530>
5,lc_attributes,<property object at 0x11f81ec50>,<property object at 0x11f81ec50>
6,lc_secrets,<property object at 0x11f81eb60>,<property object at 0x11f81eb60>
7,model_computed_fields,{},{}
8,model_config,dict (3 keys),"{'extra': 'forbid', 'protected_namespace"
9,model_extra,<property object at 0x11f203560>,<property object at 0x11f203560>


**<class 'langchain.chains.llm.LLMChain'>** — _métodos da instância_

,name,method,abstract
0,aapply,?.aapply,False
1,aapply_and_parse,?.aapply_and_parse,False
2,abatch,?.abatch,False
3,abatch_as_completed,?.abatch_as_completed,False
4,acall,?.acall,False
5,agenerate,?.agenerate,False
6,ainvoke,?.ainvoke,False
7,apply,?.apply,False
8,apply_and_parse,?.apply_and_parse,False
9,apredict,?.apredict,False


# `KWARGS`

In [ ]:
# full_init_params quer a CLASSE crua (sem parênteses, sem .method) — a
# tabela é 100% estática (assinaturas de __init__ via MRO), não depende de
# nenhuma instância. CharacterTextSplitter() ou CharacterTextSplitter.metodo
# dão AttributeError (não tem __mro__) e, mesmo funcionando, não mudariam a
# tabela — então nem vale normalizar isso na função.

_ = BaseCombineDocumentsChain

full_init_params(_)

**BaseCombineDocumentsChain** — _full init params (via MRO)_

,name,from,kind,default,required,annotation
0,data,BaseModel,VAR_KEYWORD,<class 'inspect._empty'>,False,Any


# `SIBLINGS`

In [ ]:
_ = BaseMessage

siblings(_, label=str(_))

NameError: name 'BaseMessage' is not defined

## S2 — a limitação na prática

`BaseStore` vive num módulo solto (`langchain_core/stores.py`), não num
subpacote dedicado. `siblings` sobe pro pacote inteiro `langchain_core` —
grande demais pra ser útil. Serve como lembrete: quando isso acontecer,
volte pra pista manual (import já existente, doc oficial).

In [ ]:
_ = BaseMessage

siblings(_, label=str(_))

NameError: name 'BaseMessage' is not defined